# 🔑 Semana 16 · Unidad 4 — Tablas Hash

## Información del Curso

| Aspecto | Detalle |
|--------|--------|
| **Universidad** | Universidad de Talca, Chile |
| **Carrera** | Ingeniería Civil en Informática |
| **Semestre** | 2°-3° año |
| **Curso** | Algoritmos y Estructuras de Datos |
| **Docente** | PhD. César Astudillo |
| **Clase** | Semana 16 · Unidad 4 — Tablas Hash |
| **Duración** | 90 minutos |

---
> 🎯 *Este notebook está diseñado para ser ejecutado en clase de forma interactiva.*
> *Ejecuta las celdas en orden de arriba hacia abajo.*

> ⚠️ **Aviso de evaluación.** Este tópico **NO entra en la Prueba de la Unidad 4**, que se
> rinde el viernes de esta misma semana y cubre hasta árboles balanceados. Tablas Hash sí
> entra en el **Examen Opcional Acumulativo**.

In [ ]:
# Verificación de dependencias — ejecutar primero
import sys
required = {
    'numpy': 'numpy',
    'matplotlib': 'matplotlib',
}
for nombre, paquete in required.items():
    try:
        __import__(paquete)
        print(f"✅ {nombre} instalado correctamente")
    except ImportError:
        print(f"❌ {nombre} NO encontrado — instala con: pip install {paquete}")
print("\n🐍 Python", sys.version.split()[0], "| Todo listo para comenzar.")

## 🎯 Objetivos de Aprendizaje

Al finalizar esta sesión, el estudiante será capaz de:

1. **Comprender** qué es una función hash y por qué la distribución uniforme de sus salidas
   es la condición que hace funcionar toda la estructura.
2. **Identificar** las dos estrategias clásicas de resolución de colisiones —encadenamiento
   separado y sondeo lineal— y sus modos de falla característicos.
3. **Implementar** una tabla hash completa con ambas estrategias, incluido el
   redimensionamiento y el borrado con lápidas.
4. **Analizar** el rol del factor de carga $\alpha$ y justificar por qué la tabla se duplica
   al superar un umbral, obteniendo costo esperado $O(1)$ amortizado.
5. **Comparar** tabla hash, BST y árbol balanceado como implementaciones de diccionario, y
   elegir la adecuada según el caso de uso.

# Sección 1: La idea que rompe la barrera del logaritmo (12 minutos)

## ¿Dónde quedamos?

| Implementación | `get` peor caso | ¿Orden? |
|---|---|---|
| Lista desordenada | $O(n)$ | no |
| Arreglo ordenado + búsqueda binaria | $O(\log n)$ lectura, $O(n)$ inserción | sí |
| BST simple | $O(n)$ | sí |
| **Árbol rojo-negro** | $O(\log n)$ | sí |

El rojo-negro parecía el final del camino: $O(\log n)$ garantizado. Pero todas estas
estructuras **comparan claves** para navegar, y ahí hay un límite teórico conocido.

## La idea: no buscar, calcular

> 💡 **Insight:** si la clave fuera un entero pequeño, no habría nada que buscar. La clave
> **sería** el índice del arreglo, y `get` costaría $O(1)$.

```
claves 0..5, guardadas directamente en un arreglo:

   índice:   0     1     2     3     4     5
   valor:  ['a'] ['b'] ['c'] ['d'] ['e'] ['f']

   get(3) -> arreglo[3]  ->  un solo acceso, O(1)
```

El problema es que las claves reales no son enteros pequeños: son RUTs, correos, nombres,
URLs. La solución es una función que los convierta:

> 📌 **Definición:** una **función hash** $h$ mapea una clave de un universo grande a un
> índice del rango $\{0, 1, \dots, m-1\}$, donde $m$ es el tamaño de la tabla.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Si tenemos $2^{64}$ claves posibles y una
> tabla de 1000 casillas, ¿qué es inevitable que ocurra?\"

In [ ]:
# La idea básica: calcular la posición en vez de buscarla
def hash_simple(clave, m):
    """
    Mapea una clave a un índice en [0, m).

    Usamos hash() de Python, que ya está diseñado para distribuir bien,
    y abs() porque hash() puede devolver negativos.

    Complejidad: O(1) para enteros; O(longitud) para cadenas.
    """
    return abs(hash(clave)) % m


m = 10
claves = ["ana", "luis", "carla", "diego", "sofia", "mateo"]

print(f"Tabla de m = {m} casillas\n")
print(f"{'clave':<10}{'hash(clave)':>24}{'índice':>10}")
print("-" * 46)
for k in claves:
    print(f"{k:<10}{hash(k):>24}{hash_simple(k, m):>10}")

print("\n👉 Cada clave se convierte en un índice con una sola operación.")
print("   No hay recorrido, no hay comparaciones: se CALCULA dónde está.")

## El problema inevitable: las colisiones

Con $2^{64}$ claves posibles y $m$ casillas, es **imposible** evitar que dos claves
distintas caigan en el mismo índice. Eso se llama **colisión**.

> ⚠️ **Importante:** las colisiones no son un error a evitar, son una certeza matemática a
> administrar. Todo el diseño de una tabla hash gira en torno a cómo resolverlas.

De hecho, ocurren mucho antes de lo que la intuición sugiere — es la **paradoja del
cumpleaños**: con solo 23 personas la probabilidad de que dos compartan cumpleaños ya
supera el 50%.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def prob_colision(n, m):
    """
    Probabilidad de que al insertar n claves en m casillas haya al menos una colisión.

    Complejidad: O(n)
    """
    if n > m:
        return 1.0
    p_sin = 1.0
    for i in range(n):
        p_sin *= (m - i) / m
    return 1 - p_sin


m = 365
ns = list(range(1, 101))
ps = [prob_colision(n, m) for n in ns]

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(ns, ps, linewidth=2)
ax.axhline(0.5, linestyle="--", color="red", label="50% de probabilidad")
ax.axvline(23, linestyle=":", color="gray", label="n = 23 claves")
ax.set_xlabel("Número de claves insertadas (n)")
ax.set_ylabel("P(al menos una colisión)")
ax.set_title(f"Paradoja del cumpleaños: colisiones en una tabla de m = {m} casillas")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Con solo 23 claves en 365 casillas, P(colisión) = {prob_colision(23, 365):.1%}")
print(f"Con 50 claves:  {prob_colision(50, 365):.1%}")
print(f"Con 100 claves: {prob_colision(100, 365):.1%}")
print("\n👉 Las colisiones son la norma, no la excepción. Hay que diseñar para ellas.")

# Sección 2: Encadenamiento separado (20 minutos)

## La estrategia

> 📌 **Definición:** en el **encadenamiento separado**, cada casilla de la tabla no guarda
> un par clave-valor sino una **lista** de todos los pares cuya clave haya caído ahí.

```
  índice
    0  ->  [ ]
    1  ->  [ ("ana", 21) ]
    2  ->  [ ("luis", 30) -> ("sofia", 22) ]   ← colisión: dos claves en la misma casilla
    3  ->  [ ]
    4  ->  [ ("carla", 25) ]
```

`get(k)`: se calcula $h(k)$ y se recorre **solo** la lista de esa casilla.

## El costo

Sea $n$ el número de claves y $m$ el de casillas. Definimos:

> 📌 **Definición:** el **factor de carga** es $\alpha = n/m$, el número promedio de claves
> por casilla.

Si la función hash distribuye uniformemente, cada lista tiene largo esperado $\alpha$, y

$$T_{\text{get}} = O(1 + \alpha)$$

El $1$ es calcular el hash; el $\alpha$ es recorrer la lista.

> 💡 **Insight:** si mantenemos $\alpha$ acotado por una constante —digamos $\alpha \le 0.75$—
> entonces $O(1 + \alpha) = O(1)$. **Ese es todo el truco.** Y mantener $\alpha$ acotado se
> logra duplicando la tabla cuando crece demasiado.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "¿Qué pasa con el costo si la función hash es
> pésima y manda todas las claves a la misma casilla?\"

In [ ]:
class HashEncadenamiento:
    """
    Tabla hash con resolución de colisiones por encadenamiento separado.

    Cada casilla guarda una lista de pares (clave, valor). La tabla se duplica
    cuando el factor de carga supera el umbral, para mantener las listas cortas.
    """

    def __init__(self, capacidad=8, umbral=0.75):
        self.m = capacidad          # número de casillas
        self.n = 0                  # número de claves almacenadas
        self.umbral = umbral
        self.tabla = [[] for _ in range(self.m)]
        self.colisiones = 0         # instrumentación para la clase

    def _indice(self, clave):
        """Calcula la casilla de una clave. Complejidad: O(1)"""
        return abs(hash(clave)) % self.m

    @property
    def factor_carga(self):
        """alpha = n/m, el largo promedio de las listas."""
        return self.n / self.m

    def put(self, clave, valor):
        """
        Inserta o actualiza una clave.

        Complejidad:
            Temporal: O(1 + alpha) esperado; O(n) amortizado por el redimensionamiento
            Espacial: O(1)
        """
        i = self._indice(clave)
        casilla = self.tabla[i]
        if casilla:
            self.colisiones += 1
        for pos, (k, _) in enumerate(casilla):
            if k == clave:
                casilla[pos] = (clave, valor)     # actualización, no crece n
                return
        casilla.append((clave, valor))
        self.n += 1
        if self.factor_carga > self.umbral:
            self._redimensionar(self.m * 2)

    def get(self, clave):
        """
        Busca el valor de una clave; None si no está.

        Complejidad:
            Temporal: O(1 + alpha) esperado, O(n) en el peor caso
            Espacial: O(1)
        """
        for k, v in self.tabla[self._indice(clave)]:
            if k == clave:
                return v
        return None

    def delete(self, clave):
        """
        Elimina una clave. Retorna True si estaba. Complejidad: O(1 + alpha) esperado.
        """
        casilla = self.tabla[self._indice(clave)]
        for pos, (k, _) in enumerate(casilla):
            if k == clave:
                casilla.pop(pos)     # con listas, borrar es trivial
                self.n -= 1
                return True
        return False

    def _redimensionar(self, nueva_m):
        """
        Duplica la tabla y REINSERTA todo: los índices cambian al cambiar m.

        Complejidad: O(n + m) — por eso el costo es amortizado, no constante.
        """
        viejos = [par for casilla in self.tabla for par in casilla]
        self.m = nueva_m
        self.tabla = [[] for _ in range(self.m)]
        self.n = 0
        for k, v in viejos:
            self.put(k, v)

    def __contains__(self, clave):
        return self.get(clave) is not None

    def __len__(self):
        return self.n

    def largos_de_listas(self):
        """Distribución de largos de casilla, para diagnosticar la función hash."""
        return [len(c) for c in self.tabla]


# Prueba rápida
h = HashEncadenamiento()
for i, nombre in enumerate(["ana", "luis", "carla", "diego", "sofia", "mateo", "elena"]):
    h.put(nombre, i)

print(f"Claves almacenadas : {len(h)}")
print(f"Casillas (m)       : {h.m}")
print(f"Factor de carga α  : {h.factor_carga:.2f}")
print(f"get('carla')       : {h.get('carla')}")
print(f"get('nadie')       : {h.get('nadie')}")
print(f"'sofia' in h       : {'sofia' in h}")
h.delete("luis")
print(f"tras delete('luis'): {len(h)} claves, get('luis') = {h.get('luis')}")
print(f"\nLargos de las listas: {h.largos_de_listas()}")

# Sección 3: Sondeo lineal (20 minutos)

## Otra estrategia: quedarse dentro del arreglo

El encadenamiento usa listas auxiliares. El **direccionamiento abierto** evita esa memoria
extra: si la casilla está ocupada, se busca **la siguiente libre** dentro del mismo arreglo.

> 📌 **Definición:** en el **sondeo lineal** (*linear probing*), ante una colisión en la
> posición $i$ se prueba $i+1$, luego $i+2$, y así sucesivamente módulo $m$, hasta hallar
> una casilla vacía.

```
  put("luis") -> h = 2, pero la casilla 2 está ocupada por "ana"

  índice:   0      1      2       3      4      5
          [ · ]  [ · ] ["ana"] [    ]  [ · ]  [ · ]
                          ↑       ↑
                       ocupada   aquí va "luis"
```

## El modo de falla: agrupamiento primario

> ⚠️ **Importante:** el sondeo lineal sufre **agrupamiento primario** (*primary clustering*):
> los bloques contiguos de casillas ocupadas tienden a crecer, porque cuanto más largo es un
> bloque, más probable es que la siguiente clave caiga en él y lo alargue todavía más.

El costo esperado de una búsqueda fallida es aproximadamente

$$\frac{1}{2}\left(1 + \frac{1}{(1-\alpha)^2}\right)$$

que **explota** cuando $\alpha \to 1$. Con $\alpha = 0.9$ ya son unos 50 sondeos; con
$\alpha = 0.99$, unos 5000. Por eso el sondeo lineal exige un umbral más agresivo,
típicamente $\alpha \le 0.5$.

## El problema del borrado

Con encadenamiento, borrar es sacar un elemento de una lista. Con sondeo lineal **no se
puede simplemente vaciar la casilla**: eso cortaría la cadena de sondeo y volvería
inalcanzables las claves que vienen después.

La solución es la **lápida** (*tombstone*): una marca que dice "aquí hubo algo, sigue
buscando", pero que puede reutilizarse para insertar.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Si vaciáramos la casilla en vez de poner una
> lápida, ¿qué clave concreta del ejemplo anterior se volvería inalcanzable?\"

In [ ]:
class _Lapida:
    """Marca de borrado: 'aquí hubo una clave, sigue sondeando'."""
    __repr__ = lambda self: "🪦"

LAPIDA = _Lapida()


class HashSondeoLineal:
    """
    Tabla hash con direccionamiento abierto y sondeo lineal.

    Guarda las claves en el propio arreglo. Usa lápidas para el borrado, de modo
    que las cadenas de sondeo no se corten.
    """

    def __init__(self, capacidad=8, umbral=0.5):
        self.m = capacidad
        self.n = 0                  # claves reales (sin contar lápidas)
        self.usadas = 0             # claves + lápidas: lo que ocupa la tabla
        self.umbral = umbral
        self.claves = [None] * self.m
        self.valores = [None] * self.m
        self.sondeos = 0            # instrumentación para la clase

    def _indice(self, clave):
        return abs(hash(clave)) % self.m

    @property
    def factor_carga(self):
        return self.n / self.m

    def put(self, clave, valor):
        """
        Inserta o actualiza. Complejidad: O(1) esperado si alpha se mantiene bajo.
        """
        i = self._indice(clave)
        primera_lapida = None
        while self.claves[i] is not None:
            self.sondeos += 1
            if self.claves[i] is LAPIDA:
                if primera_lapida is None:
                    primera_lapida = i        # podemos reutilizarla si la clave no existe
            elif self.claves[i] == clave:
                self.valores[i] = valor       # actualización
                return
            i = (i + 1) % self.m

        # La clave no estaba: la ponemos en la primera lápida vista, o en la casilla libre.
        destino = primera_lapida if primera_lapida is not None else i
        if primera_lapida is None:
            self.usadas += 1
        self.claves[destino] = clave
        self.valores[destino] = valor
        self.n += 1

        # Redimensionamos según las casillas USADAS: las lápidas también estorban.
        if self.usadas / self.m > self.umbral:
            self._redimensionar(self.m * 2)

    def get(self, clave):
        """
        Busca una clave. Complejidad: O(1) esperado, O(n) en el peor caso.
        """
        i = self._indice(clave)
        recorridas = 0
        while self.claves[i] is not None and recorridas < self.m:
            self.sondeos += 1
            if self.claves[i] is not LAPIDA and self.claves[i] == clave:
                return self.valores[i]
            i = (i + 1) % self.m
            recorridas += 1
        return None

    def delete(self, clave):
        """
        Borra dejando una LÁPIDA, para no cortar la cadena de sondeo.

        Complejidad: O(1) esperado.
        """
        i = self._indice(clave)
        recorridas = 0
        while self.claves[i] is not None and recorridas < self.m:
            if self.claves[i] is not LAPIDA and self.claves[i] == clave:
                self.claves[i] = LAPIDA       # NO ponemos None: rompería la cadena
                self.valores[i] = None
                self.n -= 1
                return True
            i = (i + 1) % self.m
            recorridas += 1
        return False

    def _redimensionar(self, nueva_m):
        """Reinserta todo en una tabla nueva; de paso, elimina las lápidas."""
        pares = [(k, v) for k, v in zip(self.claves, self.valores)
                 if k is not None and k is not LAPIDA]
        self.m = nueva_m
        self.claves = [None] * self.m
        self.valores = [None] * self.m
        self.n = self.usadas = 0
        for k, v in pares:
            self.put(k, v)

    def __contains__(self, clave):
        return self.get(clave) is not None

    def __len__(self):
        return self.n


# Demostración de por qué la lápida es necesaria
h = HashSondeoLineal(capacidad=8, umbral=0.9)
for nombre in ["ana", "luis", "carla", "diego"]:
    h.put(nombre, nombre.upper())

print("Estado inicial de la tabla:")
print(f"  claves: {h.claves}\n")

h.delete("ana")
print("Después de delete('ana'):")
print(f"  claves: {h.claves}")
print(f"  🪦 marca la lápida: la cadena de sondeo NO se corta.\n")

print("Las demás claves siguen siendo alcanzables:")
for nombre in ["luis", "carla", "diego"]:
    print(f"  get({nombre!r}) = {h.get(nombre)}")
print(f"  get('ana')   = {h.get('ana')}   ← correctamente borrada")

# Sección 4: El factor de carga y el redimensionamiento (18 minutos)

## Por qué hay que duplicar la tabla

Toda la promesa de $O(1)$ descansa en mantener $\alpha$ acotado. Si dejamos que $n$ crezca
con $m$ fijo, $\alpha$ crece sin límite y la tabla se degrada:

- **Encadenamiento:** el costo es $O(1+\alpha)$ — crece **linealmente** con $\alpha$.
- **Sondeo lineal:** el costo tiene $\frac{1}{(1-\alpha)^2}$ — **explota** cerca de $\alpha=1$.

## El costo amortizado

Duplicar la tabla cuesta $O(n)$: hay que reinsertar todo, porque al cambiar $m$ cambian
todos los índices.

Pero duplicamos solo cuando $n$ alcanza un múltiplo del tamaño, así que entre dos
duplicaciones hay $\Theta(n)$ inserciones baratas. El costo total de $n$ inserciones es
$O(n)$, y por lo tanto:

$$T_{\text{put amortizado}} = O(1)$$

> 💡 **Insight:** es el mismo argumento que hace $O(1)$ amortizado el `append` de una lista
> de Python. Duplicar es caro, pero tan infrecuente que se diluye.

> ⚠️ **Importante:** *amortizado* no es *garantizado*. Una inserción individual puede costar
> $O(n)$. En sistemas de tiempo real eso importa, y ahí se usan variantes de rehashing
> incremental.

Midámoslo.

In [ ]:
# Cómo se degrada cada estrategia al crecer alpha, SIN redimensionar
m_fijo = 512
alphas = [0.1, 0.25, 0.5, 0.75, 0.9, 0.95]

print(f"Tabla de m = {m_fijo} casillas, sin redimensionamiento\n")
print(f"{'α':>6} {'n':>7} {'sondeos/get enc.':>19} {'sondeos/get lineal':>21}")
print("-" * 56)

sondeos_enc, sondeos_lin = [], []
for alpha in alphas:
    n = int(alpha * m_fijo)
    claves = [f"clave_{i}" for i in range(n)]

    # Encadenamiento: el costo es el largo de la lista visitada
    enc = HashEncadenamiento(capacidad=m_fijo, umbral=99)   # umbral alto = nunca redimensiona
    for k in claves:
        enc.put(k, k)
    largos = enc.largos_de_listas()
    # Costo esperado de una búsqueda exitosa: promedio de posiciones recorridas
    coste_enc = sum(l * (l + 1) / 2 for l in largos) / max(n, 1)

    # Sondeo lineal: contamos sondeos reales
    lin = HashSondeoLineal(capacidad=m_fijo, umbral=99)
    for k in claves:
        lin.put(k, k)
    lin.sondeos = 0
    for k in claves:
        lin.get(k)
    coste_lin = lin.sondeos / max(n, 1)

    sondeos_enc.append(coste_enc)
    sondeos_lin.append(coste_lin)
    print(f"{alpha:>6.2f} {n:>7} {coste_enc:>19.2f} {coste_lin:>21.2f}")

print("\n👉 El encadenamiento se degrada suavemente (lineal en α).")
print("   El sondeo lineal se dispara cerca de α = 1: es el agrupamiento primario.")

In [ ]:
# Gráfico del costo teórico frente al medido
a = np.linspace(0.01, 0.95, 200)
teo_enc = 1 + a / 2                             # búsqueda exitosa, encadenamiento
teo_lin = 0.5 * (1 + 1 / (1 - a) ** 2)          # búsqueda fallida, sondeo lineal

fig, ax = plt.subplots(figsize=(10, 6))
ax.plot(a, teo_enc, linewidth=2, label="Encadenamiento — teórico 1 + α/2")
ax.plot(a, teo_lin, linewidth=2, label="Sondeo lineal — teórico ½(1 + 1/(1−α)²)")
ax.plot(alphas, sondeos_enc, "o", markersize=9, label="Encadenamiento — medido")
ax.plot(alphas, sondeos_lin, "s", markersize=9, label="Sondeo lineal — medido")
ax.axvline(0.75, linestyle="--", color="gray", alpha=0.7, label="umbral típico 0,75")
ax.axvline(0.5,  linestyle=":",  color="brown", alpha=0.7, label="umbral sondeo lineal 0,5")
ax.set_ylim(0, 25)
ax.set_xlabel("Factor de carga α = n/m")
ax.set_ylabel("Sondeos por operación")
ax.set_title("Por qué se redimensiona: el costo depende solo de α, no de n")
ax.legend(fontsize=9)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# El redimensionamiento en acción: alpha se mantiene acotado aunque n crezca
h = HashEncadenamiento(capacidad=8, umbral=0.75)
historia = []
for i in range(3000):
    h.put(f"clave_{i}", i)
    historia.append((i + 1, h.m, h.factor_carga))

ns = [x[0] for x in historia]
ms = [x[1] for x in historia]
als = [x[2] for x in historia]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(11, 8), sharex=True)
ax1.plot(ns, ms, linewidth=2)
ax1.set_ylabel("Casillas (m)")
ax1.set_title("La tabla duplica su tamaño: cada escalón es un redimensionamiento O(n)")
ax1.grid(alpha=0.3)

ax2.plot(ns, als, linewidth=2, color="tab:orange")
ax2.axhline(0.75, linestyle="--", color="red", label="umbral α = 0,75")
ax2.set_xlabel("Claves insertadas (n)")
ax2.set_ylabel("Factor de carga α")
ax2.set_title("…y por eso α nunca supera el umbral, sin importar cuánto crezca n")
ax2.legend()
ax2.grid(alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Tras {len(h)} inserciones: m = {h.m}, α = {h.factor_carga:.3f}")
print(f"Número de redimensionamientos: {len(set(ms)) - 1}")
print("\n👉 α se mantiene siempre acotado. Por eso get y put son O(1) esperado.")

# Sección 5: Explorador interactivo (5 minutos)

Ajusta el número de claves y el umbral, y observa cómo cambian el factor de carga, el
número de sondeos y la distribución de las casillas.

In [ ]:
# ── Explorador parametrizable ───────────────────────────────────────────────
# Cambia los valores de las llamadas de abajo y vuelve a ejecutar.

def explorar_tabla(n=200, umbral=0.75, estrategia="encadenamiento", verbose=True):
    """
    Construye una tabla hash con n claves y reporta su comportamiento.

    Parámetros:
        n (int): número de claves a insertar
        umbral (float): factor de carga que dispara el redimensionamiento
        estrategia (str): 'encadenamiento' o 'sondeo'
        verbose (bool): imprime el detalle

    Retorna:
        dict con m, alpha y la métrica de costo de la estrategia
    """
    claves = [f"clave_{i}" for i in range(n)]
    if estrategia == "encadenamiento":
        h = HashEncadenamiento(capacidad=8, umbral=umbral)
        for k in claves:
            h.put(k, k)
        largos = h.largos_de_listas()
        costo = max(largos)
        etiqueta = f"lista más larga = {costo}"
    else:
        h = HashSondeoLineal(capacidad=8, umbral=umbral)
        for k in claves:
            h.put(k, k)
        h.sondeos = 0
        for k in claves:
            h.get(k)
        costo = h.sondeos / max(n, 1)
        etiqueta = f"sondeos por get = {costo:.2f}"
    if verbose:
        print(f"n = {n:<6} umbral = {umbral:<5} {estrategia:<15} "
              f"m = {h.m:<6} α = {h.factor_carga:.3f}   {etiqueta}")
    return {"m": h.m, "alpha": h.factor_carga, "costo": costo}


print("=== Encadenamiento separado ===")
for n in (100, 500, 2000):
    explorar_tabla(n=n, umbral=0.75, estrategia="encadenamiento")

print("\n=== Sondeo lineal: el umbral importa mucho más ===")
for u in (0.5, 0.75, 0.9):
    explorar_tabla(n=500, umbral=u, estrategia="sondeo")

print("\n👉 Sube el umbral del sondeo lineal hacia 0,95 en la llamada de arriba y mira")
print("   cómo se dispara el número de sondeos: eso es el agrupamiento primario.")

## 🧪 Ejercicio 1: Contar colisiones de una función hash ⭐

**Descripción:** una función hash mala arruina la tabla. Implementa una función que, dada
una lista de claves y un tamaño de tabla `m`, cuente cuántas **casillas** reciben más de
una clave.

**Entrada:**
- `claves` (list): lista de claves
- `m` (int): número de casillas
- `fn_hash` (callable): función que recibe `(clave, m)` y retorna el índice

**Salida:** entero — cantidad de casillas con 2 o más claves.

**Ejemplo:**
```
Entrada: claves=['a','b','c'], m=1, fn_hash=lambda k,m: 0
Salida:  1     (la única casilla recibe las tres claves)
```

**Restricciones:** $1 \le m \le 10^5$.
**Complejidad esperada:** O(n + m)

In [ ]:
def contar_casillas_con_colision(claves, m, fn_hash):
    """
    Cuenta cuántas casillas reciben 2 o más claves.

    Parámetros:
        claves (list): claves a distribuir
        m (int): número de casillas
        fn_hash (callable): fn_hash(clave, m) -> índice
    Retorna:
        int: número de casillas con más de una clave
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_1(fn):
    """Ejecuta casos de prueba para el ejercicio 1."""
    import time
    h_bueno = lambda k, m: abs(hash(k)) % m
    h_pesimo = lambda k, m: 0                       # todo a la misma casilla
    h_paridad = lambda k, m: (k % 2) % m            # solo 2 destinos posibles

    casos = [
        ((['a', 'b', 'c'], 1, h_pesimo), 1, "hash pésimo: todo en una casilla"),
        ((['a'], 10, h_bueno), 0, "una sola clave, sin colisión posible"),
        (([], 10, h_bueno), 0, "sin claves"),
        ((list(range(100)), 2, h_paridad), 2, "100 enteros en 2 casillas por paridad"),
        ((list(range(10)), 10000, h_bueno), 0, "tabla enorme, colisiones improbables"),
        ((['x', 'y', 'z', 'w'], 1, h_pesimo), 1, "cuatro claves, una casilla"),
    ]
    aprobados = 0
    for (claves, m, fh), esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(list(claves), m, fh)
            t1 = time.perf_counter()
            if resultado == esperado:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms) → {resultado}")
                aprobados += 1
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado}")
                print(f"     Obtenido: {resultado}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_1(contar_casillas_con_colision)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def contar_casillas_con_colision(claves, m, fn_hash):
#     """Cuenta las casillas que reciben 2 o más claves."""
#     # Paso 1: contar cuántas claves caen en cada casilla
#     cuenta = [0] * m
#     for k in claves:
#         cuenta[fn_hash(k, m)] += 1
#     # Paso 2: contar cuántas casillas superan 1
#     return sum(1 for c in cuenta if c > 1)
#     # Complejidad: O(n + m) temporal, O(m) espacial

## 🧪 Ejercicio 2: Primer carácter que no se repite ⭐⭐

**Descripción:** dado un texto, encuentra el **primer** carácter que aparece exactamente una
vez. Es el problema clásico donde la tabla hash gana claramente: la solución ingenua es
$O(n^2)$ y con un diccionario baja a $O(n)$.

**Entrada:** `texto` (str)
**Salida:** el primer carácter no repetido, o `None` si todos se repiten.

**Ejemplo:**
```
Entrada: "algoritmos"
Salida:  'a'      (en a-l-g-o-r-i-t-m-o-s la única letra repetida es 'o';
                   'a' es la primera que aparece una sola vez)

Entrada: "aabbcc"
Salida:  None     (todas se repiten)
```

**Restricciones:** $0 \le |texto| \le 10^6$. Debe ser una sola pasada de conteo.
**Complejidad esperada:** O(n) temporal

> 💡 **Pista:** dos pasadas. La primera cuenta las apariciones; la segunda recorre el texto
> **en orden** y devuelve el primero cuyo conteo sea 1. El orden del texto es lo que da la
> respuesta, no el orden del diccionario.

In [ ]:
def primer_no_repetido(texto):
    """
    Devuelve el primer carácter que aparece exactamente una vez.

    Parámetros:
        texto (str): la cadena a analizar
    Retorna:
        str | None: el carácter, o None si no existe
    """
    # Tu código aquí
    pass

In [ ]:
def verificar_ejercicio_2(fn):
    """Ejecuta casos de prueba para el ejercicio 2."""
    import time
    casos = [
        ("algoritmos", "a", "caso normal"),
        ("aabbcc", None, "todos repetidos"),
        ("", None, "cadena vacía"),
        ("x", "x", "un solo carácter"),
        ("aabbc", "c", "el único no repetido está al final"),
        ("swiss", "w", "el primero no repetido no es el primer carácter"),
        ("a" * 100000 + "b", "b", "cadena grande: debe ser O(n), no O(n²)"),
    ]
    aprobados = 0
    for texto, esperado, desc in casos:
        t0 = time.perf_counter()
        try:
            resultado = fn(texto)
            t1 = time.perf_counter()
            lento = (t1 - t0) > 1.0
            if resultado == esperado and not lento:
                print(f"  ✅ {desc} ({(t1-t0)*1000:.2f}ms) → {resultado!r}")
                aprobados += 1
            elif lento:
                print(f"  ❌ {desc} — demasiado lento ({(t1-t0):.2f}s): ¿estás usando count() dentro del bucle?")
            else:
                print(f"  ❌ {desc}")
                print(f"     Esperado: {esperado!r}")
                print(f"     Obtenido: {resultado!r}")
        except Exception as e:
            print(f"  💥 {desc} — Error: {e}")
    print(f"\n{'🎉 Todos los casos pasaron!' if aprobados == len(casos) else f'⚠️  {aprobados}/{len(casos)} casos correctos'}")

verificar_ejercicio_2(primer_no_repetido)

In [ ]:
# ═══════════════════════════════════════════════════
# SOLUCIÓN — Descomenta para ver después de intentarlo
# ═══════════════════════════════════════════════════

# def primer_no_repetido(texto):
#     """Dos pasadas: contar con un diccionario, luego recorrer en orden."""
#     # Paso 1: contar apariciones — O(n), cada acceso al dict es O(1) esperado
#     conteo = {}
#     for ch in texto:
#         conteo[ch] = conteo.get(ch, 0) + 1
#     # Paso 2: recorrer el TEXTO en orden (no el dict) y devolver el primero con conteo 1
#     for ch in texto:
#         if conteo[ch] == 1:
#             return ch
#     # Paso 3: si ninguno tiene conteo 1
#     return None
#     # Complejidad: O(n) temporal, O(k) espacial con k = alfabeto distinto
#
# NOTA: la versión ingenua sería  for ch in texto: if texto.count(ch) == 1: return ch
# que es O(n²) — con la cadena de 100.001 caracteres del test, no termina.

## 🔬 Zona de Experimentación

Las siguientes celdas son tuyas para experimentar. Algunas sugerencias:
- Define una función hash deliberadamente mala (por ejemplo, `len(clave) % m`) y mide cómo
  se degrada `get` en `HashEncadenamiento`.
- Sube el umbral de `HashSondeoLineal` a 0,95 e inserta 10.000 claves. ¿Cuántos sondeos
  por `get` necesitas?
- Inserta y borra 10.000 veces alternadamente en la tabla con sondeo lineal. ¿Qué pasa con
  las lápidas si nunca se dispara un redimensionamiento?

In [ ]:
# Espacio libre para experimentar
# Sugerencia: hash_malo = lambda k, m: len(str(k)) % m   ¿qué le pasa a la tabla?


In [ ]:
# Espacio libre para experimentar
# Sugerencia: compara los tiempos de HashEncadenamiento contra el dict nativo.


## ✍️ Autoevaluación

Este contenido entra en el **Examen Opcional Acumulativo**, no en la Prueba de la Unidad 4.

## ✍️ Autoevaluación

Responde cada pregunta antes de abrir la respuesta.

**1. ¿Cuál es el costo esperado de get en una tabla con encadenamiento separado?**

- a) O(1 + α)
- b) O(log n)
- c) O(n)
- d) O(α²)

<details>
<summary>Ver respuesta</summary>

**a) O(1 + α)** — O(1) por calcular el hash más O(α) por recorrer la lista de esa casilla. Si α se mantiene acotado, el total es O(1).

</details>

---

**2. ¿Por qué el borrado en sondeo lineal usa una lápida en vez de vaciar la casilla?**

- a) Para ahorrar memoria
- b) Porque vaciarla cortaría la cadena de sondeo y volvería inalcanzables claves posteriores
- c) Porque el borrado real es demasiado lento
- d) Para poder deshacer el borrado más tarde

<details>
<summary>Ver respuesta</summary>

**b) Porque vaciarla cortaría la cadena de sondeo y volvería inalcanzables claves posteriores** — La búsqueda se detiene al encontrar una casilla vacía. Si vaciáramos la casilla borrada, toda clave que hubiera sondeado más allá quedaría inaccesible.

</details>

---

**3. ¿Qué le ocurre al sondeo lineal cuando α se acerca a 1?**

- a) Nada, el costo sigue siendo O(1)
- b) El costo crece linealmente con α
- c) El costo explota por el término 1/(1−α)²: agrupamiento primario
- d) La tabla deja de aceptar claves nuevas

<details>
<summary>Ver respuesta</summary>

**c) El costo explota por el término 1/(1−α)²: agrupamiento primario** — Los bloques contiguos de casillas ocupadas se realimentan: cuanto más largos son, más probable es que crezcan. Por eso el umbral del sondeo lineal es más bajo, ~0,5.

</details>

---

**4. El redimensionamiento cuesta O(n). ¿Por qué se dice que put es O(1)?**

- a) Porque el redimensionamiento es en realidad O(1)
- b) Porque es un costo AMORTIZADO: duplicar es raro y se diluye entre muchas inserciones baratas
- c) Porque la tabla nunca se redimensiona en la práctica
- d) Porque O(n) y O(1) son equivalentes para n pequeño

<details>
<summary>Ver respuesta</summary>

**b) Porque es un costo AMORTIZADO: duplicar es raro y se diluye entre muchas inserciones baratas** — Entre dos duplicaciones hay Θ(n) inserciones baratas, así que n inserciones cuestan O(n) en total: O(1) amortizado. Ojo: amortizado no es garantizado.

</details>

---

**5. Necesitas responder «¿qué claves hay entre 100 y 200?». ¿Qué estructura eliges?**

- a) Tabla hash: es O(1)
- b) Árbol rojo-negro: mantiene el orden y responde el rango en O(log n + k)
- c) Da lo mismo, ambas sirven
- d) Lista desordenada

<details>
<summary>Ver respuesta</summary>

**b) Árbol rojo-negro: mantiene el orden y responde el rango en O(log n + k)** — La tabla hash destruye el orden a propósito: para un rango tendría que recorrer las m casillas, O(n). El árbol balanceado mantiene el orden y poda la búsqueda.

</details>

# Sección 6: Cierre de la Unidad 4 (10 minutos)

## Las cinco implementaciones de diccionario

| | Lista desordenada | Arreglo ordenado | BST simple | Rojo-negro | **Tabla hash** |
|---|---|---|---|---|---|
| `get` promedio | $O(n)$ | $O(\log n)$ | $O(\log n)$ | $O(\log n)$ | **$O(1)$** |
| `get` **peor caso** | $O(n)$ | $O(\log n)$ | $O(n)$ | **$O(\log n)$** | $O(n)$ |
| `put` promedio | $O(n)$ | $O(n)$ | $O(\log n)$ | $O(\log n)$ | **$O(1)$** |
| `put` peor caso | $O(n)$ | $O(n)$ | $O(n)$ | **$O(\log n)$** | $O(n)$ |
| Mantiene el orden | no | **sí** | **sí** | **sí** | **no** |
| Consultas de rango | $O(n)$ | $O(\log n + k)$ | $O(\log n + k)$ | $O(\log n + k)$ | $O(n)$ |
| `min` / `max` / `rank` | $O(n)$ | $O(1)$ / $O(\log n)$ | $O(\log n)$ | $O(\log n)$ | $O(n)$ |

## Cómo elegir

> 📌 **La regla práctica:**
> - ¿Solo necesitas `get`, `put` y `delete` por clave exacta? → **tabla hash**.
> - ¿Necesitas orden: rangos, mínimos, `rank`, recorrido ordenado? → **árbol balanceado**.
> - ¿Necesitas garantía de peor caso, no solo promedio? → **árbol balanceado**.

En Python: `dict` y `set` son tablas hash. Si necesitas orden, la biblioteca estándar no
trae un árbol balanceado; se usa una lista ordenada con el módulo `bisect`, o una
implementación propia como la de la clase pasada.

> ⚠️ **Importante:** el $O(1)$ de la tabla hash es *esperado*, y depende de que la función
> hash distribuya bien. Un atacante que conozca la función puede fabricar claves que colisionen
> todas y degradar el servicio a $O(n)$ — es un ataque real, y por eso Python aleatoriza el
> hash de las cadenas en cada ejecución.

> 🎙️ **[PAUSA PROFESOR]** Pregunta sugerida: "Ejecuten `hash('hola')` en dos terminales de
> Python distintas. ¿Por qué dan resultados diferentes?"

## Lo que aprendimos hoy

1. Una **función hash** convierte una clave cualquiera en un índice: se **calcula** la
   posición en vez de buscarla.
2. Las **colisiones son inevitables** y ocurren mucho antes de lo intuitivo (paradoja del
   cumpleaños).
3. **Encadenamiento separado:** una lista por casilla, costo $O(1+\alpha)$, degradación suave.
4. **Sondeo lineal:** todo dentro del arreglo, sufre agrupamiento primario y necesita
   lápidas para borrar.
5. El **factor de carga $\alpha$** lo gobierna todo; el redimensionamiento lo mantiene
   acotado y da $O(1)$ **amortizado**.
6. La tabla hash cambia **orden por velocidad**. Si necesitas el orden, el árbol balanceado
   sigue siendo la respuesta.

## 📚 Lecturas Recomendadas y Práctica

### Textbooks

| Libro | Edición | Capítulo | Tema |
|-------|---------|----------|------|
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 11 | Tablas hash, encadenamiento y direccionamiento abierto |
| Cormen et al. (CLRS) — *Introduction to Algorithms* | 4ª ed. | Cap. 11.3 | Funciones hash y hashing universal |
| Goodrich, Tamassia & Goldwasser (GTG) — *Data Structures and Algorithms in Python* | 1ª ed. | Cap. 10.2 | Hash tables en Python |
| Miller & Ranum (M&R) — *Problem Solving with Algorithms and Data Structures Using Python* | 2011 | Cap. 5.5 | Hashing |
| Bhargava (Grok) — *Grokking Algorithms* | 2ª ed. | Cap. 5 | Hash tables, con intuición visual |
| Ramalho (Fluent) — *Fluent Python* | 2ª ed. | Cap. 3 | Cómo funciona `dict` por dentro en CPython |

### Recursos gratuitos en línea

- 🌐 [VisuAlgo — Hash Table](https://visualgo.net/en/hashtable) — encadenamiento y sondeo lineal animados.
- 🌐 [Open Hashing / Closed Hashing (USFCA)](https://www.cs.usfca.edu/~galles/visualization/OpenHash.html) — inserta claves y observa las colisiones.
- 📄 [Python `dict` implementation notes](https://github.com/python/cpython/blob/main/Objects/dictobject.c) — comentarios del código fuente de CPython.

### Práctica en Codeforces (soporta Python 3)

> 🔍 **Cómo filtrar:** ve a [codeforces.com/problemset](https://codeforces.com/problemset),
> escribe `data structures` o `implementation` en **Tags** y ajusta **Rating**.

**Escala de dificultad orientativa para este curso:**

| Rating | Nivel | Descripción |
|--------|-------|-------------|
| 800 | ⭐ | Aplicación directa — la mayoría puede resolverlo |
| 1000–1200 | ⭐⭐ | Requiere una pequeña adaptación |
| 1300+ | ⭐⭐⭐ | Combina la idea con otra — desafío |

**Problemas recomendados para este tópico:**

| # | Problema | Rating | Por qué es útil |
|---|----------|--------|-----------------|
| 1 | [4C — Registration System](https://codeforces.com/problemset/problem/4/C) | ⭐ 1300 | El caso de uso canónico de un diccionario: contar apariciones por clave |
| 2 | [977C — Less or Equal](https://codeforces.com/problemset/problem/977/C) | ⭐⭐ 1200 | Contrasta cuándo basta un hash y cuándo hace falta el orden |
| 3 | [1005C — Summarize to the Power of Two](https://codeforces.com/problemset/problem/1005/C) | ⭐⭐ 1300 | Búsqueda de complementos con un diccionario: O(n log C) en vez de O(n²) |
| 4 | [1189B — Number Circle](https://codeforces.com/problemset/problem/1189/B) | ⭐⭐ 1100 | Requiere decidir entre ordenar y usar conteo por clave |

⚠️ Los problemas 1 y 3 son el **mínimo esperado**. Los demás son desafío opcional.